In [130]:
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [131]:

X = df.drop(columns=['quality'])
y = df['quality']

standard_scaler = StandardScaler()
X = standard_scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)




### First, I'll use all features with the 'Support Vector Machine' (RBF and Linear kernels), applying both 'One-vs-Rest' and 'One-vs-One' strategies. I will also evaluate the variance and bias to assess the base model's performance initially also I'll use cross-Validation to make sure the performance holds across different train-test splits.

In [132]:
def check_bias_variance(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

    accuracy_diff = train_accuracy - test_accuracy

    if abs(accuracy_diff) > 0.05:  # 5% threshold
        if train_accuracy > test_accuracy:
            print("Potential high variance (overfitting)\n")
        else:
            print("Potential high bias (underfitting)\n")

    elif train_accuracy < 0.7 and test_accuracy < 0.7:  # 70% threshold
        print("Model is balanced but has low overall accuracy (potential underfitting)\n")
    else:
        print("Model is balanced\n")


In [133]:
# Linear
ovr_model_linear = OneVsRestClassifier(SVC(kernel='linear'))

ovr_model_linear.fit(X_train, y_train)

cv_results = cross_validate(OneVsRestClassifier(SVC(kernel='linear')), X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier LINEAR:')
check_bias_variance(train_accuracy, test_accuracy)


OneVsRestClassifier LINEAR:
Training Accuracy: 56.927%
Testing Accuracy: 54.875%
Model is balanced but has low overall accuracy (potential underfitting)



In [134]:
#RBF
ovr_model_rbf = OneVsRestClassifier(SVC(kernel='rbf'))

ovr_model_rbf.fit(X_train, y_train)

cv_results = cross_validate(OneVsRestClassifier(SVC(kernel='rbf')), X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)

OneVsRestClassifier RBF:
Training Accuracy: 71.193%
Testing Accuracy: 58.369%
Potential high variance (overfitting)



In [135]:
from sklearn.multiclass import OneVsOneClassifier

# Linear
ovo_model_linear = OneVsOneClassifier(SVC(kernel='linear'))

ovo_model_linear.fit(X_train, y_train)

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='linear')), X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
check_bias_variance(train_accuracy, test_accuracy)

OneVsOneClassifier LINEAR : 
Training Accuracy: 61.557%
Testing Accuracy: 57.751%
Model is balanced but has low overall accuracy (potential underfitting)



In [136]:
# RBF
ovo_model_rbf = OneVsOneClassifier(SVC(kernel='rbf'))

ovo_model_rbf.fit(X_train, y_train)

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='rbf')), X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)


OneVsOneClassifier RBF:
Training Accuracy: 69.633%
Testing Accuracy: 58.507%
Potential high variance (overfitting)



### Second : As I've noticed there are models overfitting and there are models almost balanced but the accuracy is still under the target which is more than 70%, so I'll go with the selected Features from the data preprocessing to see if the accuracy gets better and the model get's more balanced.  

In [173]:
'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)


### Third : I'll do the first step again but with the selected features...

In [174]:
# Linear
ovr_model_linear = OneVsRestClassifier(SVC(kernel='linear'))

ovr_model_linear.fit(X_train, y_train)

cv_results = cross_validate(OneVsRestClassifier(SVC(kernel='linear')), X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier LINEAR:')
check_bias_variance(train_accuracy, test_accuracy)

OneVsRestClassifier LINEAR:
Training Accuracy: 54.236%
Testing Accuracy: 52.678%
Model is balanced but has low overall accuracy (potential underfitting)



In [175]:
#RBF
ovr_model_rbf = OneVsRestClassifier(SVC(kernel='rbf'))

ovr_model_rbf.fit(X_train, y_train)

cv_results = cross_validate(OneVsRestClassifier(SVC(kernel='rbf')), X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)

OneVsRestClassifier RBF:
Training Accuracy: 65.998%
Testing Accuracy: 59.671%
Potential high variance (overfitting)



In [176]:
# Linear
ovo_model_linear = OneVsOneClassifier(SVC(kernel='linear'))

ovo_model_linear.fit(X_train, y_train)

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='linear')), X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
check_bias_variance(train_accuracy, test_accuracy)

OneVsOneClassifier LINEAR : 
Training Accuracy: 58.848%
Testing Accuracy: 57.618%
Model is balanced but has low overall accuracy (potential underfitting)



In [177]:
# RBF
ovo_model_rbf = OneVsOneClassifier(SVC(kernel='rbf'))

ovo_model_rbf.fit(X_train, y_train)

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='rbf')), X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF:')
check_bias_variance(train_accuracy, test_accuracy)


OneVsOneClassifier RBF:
Training Accuracy: 64.798%
Testing Accuracy: 59.397%
Potential high variance (overfitting)



### Fourth : we still see models that overfitting before the selection still overfits with the selected features but the accuracy slightly got better, so I'll go with Hyperparameters tuning only on SVM(OneVsOne Linear) as the model is almost balanced and the accuracy (57%) is not that differ from the RBF's accuracy (59%) but they overfit, to increase the accuracy. 

In [ ]:
''' 
OneVsOneClassifier LINEAR : 
Training Accuracy: 58.848%
Testing Accuracy: 57.618%
Model is balanced but has low overall accuracy (potential underfitting)
'''